# Week 09 · 可靠性、分层与异步

路由层处理 HTTP，服务层编排业务，仓储层访问数据，检索器负责召回，模型适配器负责外部协议。分层的价值是让变化局部化，不是为每个函数都造一个抽象类。Protocol 可以描述接口，测试用内存实现替代网络边界。

异步用于等待 I/O，不自动加速 CPU 密集运算。阻塞解析需要线程/进程或后台任务。timeout 限制等待，取消必须传播到正在运行的工作。重试只适用于瞬时、可重放错误；校验失败、鉴权失败不应不断重试。记录请求 ID、阶段和耗时，避免原始密钥进入日志。

上传限制同时检查字节数、扩展名、解析结果和资源消耗。仅检查后缀不能保证内容安全。Notebook 本质可执行任意管理员代码，不是恶意代码沙箱；所以网站绝不允许访客执行。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
import asyncio,json,time,logging,io
from typing import Protocol
from dataclasses import dataclass
from unittest.mock import AsyncMock

class Retriever(Protocol):
    async def search(self,query:str)->list[str]: ...
@dataclass
class QAService:
    retriever: Retriever
    async def answer(self,query:str)->dict:
        if not query.strip():raise ValueError("empty query")
        # wait_for 会取消超时协程；真实适配器也要处理资源清理。
        passages=await asyncio.wait_for(self.retriever.search(query),timeout=.2)
        return {"answer":passages[0] if passages else "Insufficient evidence", "count":len(passages)}

async def retry_read(operation,attempts=3):
    for attempt in range(attempts):
        try:return await operation()
        except TimeoutError:
            if attempt==attempts-1:raise
            await asyncio.sleep(.01*2**attempt)

async def verify_service():
    retriever=AsyncMock();retriever.search.return_value=["A transaction is atomic."]
    service=QAService(retriever)
    assert (await service.answer("transaction"))["count"]==1
    retriever.search.assert_awaited_once_with("transaction")
    retriever.search.return_value=[]
    assert (await service.answer("unknown"))["answer"]=="Insufficient evidence"
    operation=AsyncMock(side_effect=[TimeoutError(),"success"])
    assert await retry_read(operation)=="success"
    assert operation.await_count==2
    async def slow(_):await asyncio.sleep(1)
    retriever.search.side_effect=slow
    try:await service.answer("slow")
    except TimeoutError:print("超时被正确传播")
    return {"event":"service_checks","success":True}

# Notebook 已有事件循环，所以使用 await；脚本入口才使用 asyncio.run。
print(json.dumps(await verify_service()))
def validate_upload(name,content):
    if len(content)>1024:raise ValueError("file too large for this demo")
    if not name.endswith((".txt",".md")):raise ValueError("unsupported type")
    return content.decode("utf-8")
assert validate_upload("notes.md",b"hello")=="hello"

## 练习 / Exercises
补充空输入、解码失败、超出大小三个测试。把日志中 token/password 字段替换为 [redacted]。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
def redact(fields):
    return {k:"[redacted]" if k.lower() in {"token","password","api_key"} else v for k,v in fields.items()}
assert redact({"token":"never-log-me","request_id":"r1"})=={"token":"[redacted]","request_id":"r1"}
for name,data in [("x.exe",b"a"),("x.txt",b"x"*1025),("x.txt",b"\xff")]:
    try:validate_upload(name,data)
    except (ValueError,UnicodeDecodeError):print("预期拒绝",name,len(data))